## 5. Entraînement YOLOv9-C (`train_dual.py`)

In [ ]:
from pathlib import Path
import subprocess, sys

YOLO_DIR = Path("../yolov9").resolve()
RUN_NAME = "yolov9c_vehicles_e10"
RUN_DIR = YOLO_DIR / "runs/train" / RUN_NAME
DEVICE = "0"  # Mettre "cpu" si pas de GPU

cmd = [
    sys.executable, str(YOLO_DIR / "train_dual.py"),
    "--weights", str(YOLO_DIR / "yolov9-c.pt"),
    "--cfg", str(YOLO_DIR / "models/detect/yolov9-c.yaml"),
    "--data", str(Path("../trafic_data/data_1.yaml").resolve()),
    "--hyp", str(YOLO_DIR / "data/hyps/hyp.scratch-high.yaml"),
    "--img", "640", "--batch-size", "4", "--epochs", "10",
    "--device", DEVICE, "--workers", "4",
    "--project", str(YOLO_DIR / "runs/train"), "--name", RUN_NAME, "--exist-ok",
]
print("Commande :", " ".join(cmd))

In [ ]:
# Lancer l'entraînement sur GPU
res = subprocess.run(cmd, cwd=str(YOLO_DIR))
print("Code de retour :", res.returncode)

## Suivi des métriques et checkpoints (`best.pt`, `last.pt`)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

best_pt = RUN_DIR / "weights/best.pt"
last_pt = RUN_DIR / "weights/last.pt"
print("best.pt existe :", best_pt.exists())
print("last.pt existe :", last_pt.exists())

df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]
print(df.tail(5).to_string(index=False))

df.plot(x="epoch", y=["metrics/precision", "metrics/recall", "metrics/mAP_0.5", "metrics/mAP_0.5:0.95"], marker="o", figsize=(9, 4))
plt.title("Métriques de validation")
plt.show()

- **Fine-tuning :** on part des poids `yolov9-c.pt` (COCO) et on réentraîne toutes les couches sur nos 21 classes.
- **Époque / Batch :** 10 passages complets sur les données, par lots de 4 images.
- **`best.pt` / `last.pt` :** meilleur modèle sur `valid` / modèle de la dernière époque.